# EnergyInvest — Colab Demo

**PE6201 · Emerging AI Technologies · Final Project**

This notebook reproduces the main EnergyInvest workflow using the processed **Global Integrated Power Tracker (GIPT), September 2026** data included in the project repository.

### What this notebook demonstrates
1. Upload and unpack the EnergyInvest repository.
2. Install dependencies.
3. Verify the processed GIPT data.
4. Retrain the XGBoost model.
5. Reproduce the evaluation metrics.
6. Compare EnergyInvest with `Always Low` and `Always High` baselines.
7. Inspect subgroup recall for confirmed vs inferred outcomes.
8. Display a confusion matrix.
9. Show example predictions for unresolved projects.

> **Responsible-use boundary:** EnergyInvest is a first-pass triage tool. It does not make investment or capital-allocation decisions.


## 1. Upload the final repository ZIP

Upload:

`EnergyInvest_GIPT_Final_Submission.zip`

The next cell automatically extracts the repository and moves Colab into the correct project folder.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
import os

uploaded = files.upload()

zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
if zip_name is None:
    raise ValueError("Please upload the EnergyInvest final repository ZIP file.")

extract_dir = Path("/content/energyinvest_demo")
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(extract_dir)

repo_candidates = [
    p.parent for p in extract_dir.rglob("requirements.txt")
    if (p.parent / "config.py").exists()
]

if not repo_candidates:
    raise FileNotFoundError("Could not locate the EnergyInvest repository after extraction.")

repo = repo_candidates[0]
os.chdir(repo)

print("Repository ready:", repo)
print("Current working directory:", Path.cwd())


## 2. Install project dependencies

In [ ]:
!pip install -q -r requirements.txt
print("Dependencies installed.")


## 3. Verify the processed GIPT dataset

The repository already contains the processed modelling table, so the original Excel workbook is **not required** for this demo.


In [ ]:
import pandas as pd
from config import DATA_PATH, FEATURES, STATUS_COL, GROUP_COL

df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Processed rows: {len(df):,}")
print(f"Feature count: {len(FEATURES)}")

print("\nFeatures used:")
for feature in FEATURES:
    print(" -", feature)

print("\nTop project statuses:")
display(df[STATUS_COL].value_counts().to_frame("count").head(12))

print("\nMissing GEM location ID rate:")
print(f"{df[GROUP_COL].isna().mean():.2%}")


## 4. Retrain EnergyInvest

This runs the repository's actual pipeline:

- create resolved historical labels;
- make a grouped holdout by `GEM location ID`;
- train XGBoost;
- apply the abstention band;
- save evaluation metrics and holdout predictions;
- score unresolved projects.


In [ ]:
!python -m src.train


## 5. Reproduce the main evaluation results

In [ ]:
import json
import pandas as pd
from config import METRICS_PATH, PREDICTIONS_PATH

with open(METRICS_PATH, "r", encoding="utf-8") as f:
    metrics = json.load(f)

summary = pd.DataFrame({
    "Metric": [
        "High-risk recall",
        "Flag rate",
        "Workload reduction",
        "Precision among flagged",
        "Abstention rate",
        "Confirmed cancelled recall",
        "Cancelled-inferred recall",
        "Shelved recall",
        "Shelved-inferred recall",
    ],
    "Result": [
        metrics["model"]["recall_high_risk"],
        metrics["model"]["flag_rate"],
        metrics["model"]["workload_reduction"],
        metrics["model"]["precision_flagged"],
        metrics["model"]["abstention_rate"],
        metrics["model"]["recall_cancelled"],
        metrics["model"]["recall_cancelled_inferred_4y"],
        metrics["model"]["recall_shelved"],
        metrics["model"]["recall_shelved_inferred_2y"],
    ]
})

summary["Result"] = summary["Result"].map(lambda x: f"{x:.2%}")
display(summary)

print("\nDataset sizes")
print(f"Resolved records: {metrics['data']['n_resolved']:,}")
print(f"Training records: {metrics['data']['n_train']:,}")
print(f"Holdout records: {metrics['data']['n_test']:,}")


## 6. Compare against the two lazy baselines

Recall alone is not enough:

- **Always Low** saves all review work but misses every risky project.
- **Always High** gets 100% recall but sends every project to the analyst.


In [ ]:
comparison = pd.DataFrame([
    {
        "Approach": "Always Low",
        "Recall": metrics["baselines"]["always_low"]["recall"],
        "Flag Rate": metrics["baselines"]["always_low"]["flag_rate"],
        "Workload Reduction": metrics["baselines"]["always_low"]["workload_reduction"],
    },
    {
        "Approach": "Always High",
        "Recall": metrics["baselines"]["always_high"]["recall"],
        "Flag Rate": metrics["baselines"]["always_high"]["flag_rate"],
        "Workload Reduction": metrics["baselines"]["always_high"]["workload_reduction"],
    },
    {
        "Approach": "EnergyInvest",
        "Recall": metrics["model"]["recall_high_risk"],
        "Flag Rate": metrics["model"]["flag_rate"],
        "Workload Reduction": metrics["model"]["workload_reduction"],
    },
])

for col in ["Recall", "Flag Rate", "Workload Reduction"]:
    comparison[col] = comparison[col].map(lambda x: f"{x:.2%}")

display(comparison)


## 7. Confirmed vs inferred outcome check

GEM's `cancelled - inferred 4 y` and `shelved - inferred 2 y` statuses partly encode source-side heuristics. Reporting subgroup recall separately helps test whether model performance depends disproportionately on inferred cases.


In [ ]:
subgroup = pd.DataFrame({
    "Historical outcome": [
        "cancelled",
        "cancelled - inferred 4 y",
        "shelved",
        "shelved - inferred 2 y",
    ],
    "Recall": [
        metrics["model"]["recall_cancelled"],
        metrics["model"]["recall_cancelled_inferred_4y"],
        metrics["model"]["recall_shelved"],
        metrics["model"]["recall_shelved_inferred_2y"],
    ]
})
subgroup["Recall"] = subgroup["Recall"].map(lambda x: f"{x:.2%}")
display(subgroup)


## 8. Confusion matrix

`Needs Manual Review` counts as an **analyst flag** because it still requires human attention.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cm = np.array(metrics["model"]["confusion_matrix"])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm)

ax.set_xticks([0, 1], labels=["Not flagged", "Flagged"])
ax.set_yticks([0, 1], labels=["Historical low risk", "Historical high risk"])
ax.set_xlabel("EnergyInvest triage")
ax.set_ylabel("Historical outcome")
ax.set_title("EnergyInvest Holdout Confusion Matrix")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center")

plt.colorbar(im, ax=ax)
plt.show()


## 9. Inspect row-level holdout predictions

In [ ]:
pred = pd.read_csv(PREDICTIONS_PATH)

display(
    pred[
        [
            "GEM unit/phase ID",
            "GEM location ID",
            "Status",
            "target_high_risk",
            "risk_score",
            "triage",
            "analyst_flag",
        ]
    ].head(15)
)


## 10. Show unresolved projects scored by EnergyInvest

These are records currently marked `announced`, `pre-construction`, or `construction`. They are not used as historical labels; they illustrate the intended deployment population.


In [ ]:
from pathlib import Path

unresolved_path = Path("outputs/unresolved_scored.csv")
unresolved = pd.read_csv(unresolved_path, low_memory=False)

print(f"Unresolved projects scored: {len(unresolved):,}")
print("\nTriage distribution:")
display(unresolved["triage"].value_counts().to_frame("count"))

sample_cols = [
    "GEM unit/phase ID",
    "Plant / Project name",
    "Country/area",
    "Status",
    "Capacity (MW)",
    "risk_score",
    "triage",
]
sample_cols = [c for c in sample_cols if c in unresolved.columns]

display(
    unresolved.sort_values("risk_score", ascending=False)[sample_cols].head(10)
)


## 11. Translate flag rate into analyst workload

For a hypothetical weekly pipeline of **50 projects**, estimate how many still require analyst attention.


In [ ]:
pipeline_size = 50
flag_rate = metrics["model"]["flag_rate"]

projects_to_open = pipeline_size * flag_rate
projects_avoided = pipeline_size - projects_to_open

print(f"Pipeline size: {pipeline_size}")
print(f"Estimated projects requiring attention: {projects_to_open:.1f} (~{round(projects_to_open)})")
print(f"Estimated first-pass reviews avoided: {projects_avoided:.1f}")
print(f"Workload reduction: {metrics['model']['workload_reduction']:.2%}")


## 12. Final interpretation

The result should be interpreted as a **risk-capture vs workload trade-off**, not as evidence that the model can replace due diligence.

- High-risk recall exceeds the original 80% target.
- Flag rate is far below 100%, producing meaningful workload reduction.
- Precision remains modest, so many flagged records are false positives.
- Some high-risk records are still missed.
- The current GIPT release is a snapshot, so this demonstrates cross-facility generalisation rather than true forward-in-time prediction.

**Future priority:** validate on multiple dated GIPT releases, calibrate probabilities, and tune thresholds systematically before considering real deployment.
